In [1]:
import pandas as pd

df = pd.read_excel('../data/raw/kaggle_ipo_raw.xlsx')

print(df.shape)
df.head()



(652, 13)


,Date,IPO_Name,Issue_Size(crores),QIB,HNI,RII,Total,Offer Price,List Price,Listing Gain,CMP(BSE),CMP(NSE),Current Gains
0,2026-05-14,Bagmane Prime Offi...,3405.00,17.09,15.79,0.00,16.50,100,103.4,3.40,103.92,103.97,3.92
1,2026-05-08,Onemi Technology S...,925.92,24.87,6.57,2.03,9.50,171,191.0,11.70,231.70,231.74,35.50
2,2026-04-29,Citius Transnet In...,1105.00,8.54,11.77,0.00,10.01,100,104.5,4.50,105.87,106.04,5.87
3,2026-04-17,Om Power Transmiss...,150.06,3.65,7.06,1.54,3.33,175,181.1,3.49,190.80,190.74,9.03
4,2026-04-02,Sai Parenteral's L...,408.79,1.71,2.36,0.12,1.05,392,405.0,3.32,487.10,486.70,24.26


In [2]:
df.info()
df['Date'].dt.year.value_counts().sort_index()

<class 'pandas.DataFrame'>
RangeIndex: 652 entries, 0 to 651
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                652 non-null    datetime64[us]
 1   IPO_Name            652 non-null    str           
 2   Issue_Size(crores)  652 non-null    float64       
 3   QIB                 650 non-null    float64       
 4   HNI                 650 non-null    float64       
 5   RII                 650 non-null    float64       
 6   Total               650 non-null    float64       
 7   Offer Price         652 non-null    int64         
 8   List Price          652 non-null    float64       
 9   Listing Gain        650 non-null    float64       
 10  CMP(BSE)            650 non-null    float64       
 11  CMP(NSE)            642 non-null    float64       
 12  Current Gains       649 non-null    float64       
dtypes: datetime64[us](1), float64(10), int64(1), str(1)
memory us

Date
2010     74
2011     39
2012     11
2013      4
2014      6
2015     20
2016     27
2017     38
2018     24
2019     17
2020     16
2021     65
2022     39
2023     60
2024     82
2025    107
2026     23
Name: count, dtype: int64

In [3]:
df['IPO_Name'].head(20).tolist()

['Bagmane Prime Offi...',
 'Onemi Technology S...',
 'Citius Transnet In...',
 'Om Power Transmiss...',
 "Sai Parenteral's L...",
 'Powerica Ltd',
 'Amir Chand Jagdish...',
 'Central Mine Plann...',
 'GSP Crop Science L...',
 'RaajMarg Infra Inv...',
 'Innovision Ltd',
 'Rajputana Stainles...',
 'Sedemac Mechatroni...',
 'Omnitech Engineeri...',
 'PNGS Reva Diamond ...',
 'Clean Max Enviro E...',
 'Shree Ram Twistex ...',
 'Gaudium IVF and Wo...',
 'Aye Finance Ltd',
 'Fractal Analytics ...']

In [4]:
#filtering to remove extra years
df_filtered = df[(df['Date'].dt.year >=2018) & (df['Date'].dt.year <= 2025)].copy()
#dropping columns which aren't of any use
df_filtered = df_filtered.drop(columns = ['CMP(BSE)','CMP(NSE)','Current Gains'])
df_filtered = df_filtered.reset_index(drop=True)
print(df_filtered.shape)
df_filtered['Date'].dt.year.value_counts().sort_index()

(410, 10)


Date
2018     24
2019     17
2020     16
2021     65
2022     39
2023     60
2024     82
2025    107
Name: count, dtype: int64

In [5]:
df_filtered = df_filtered.rename(columns ={ 'Date' : 'listing_date',
                                            'IPO_Name' : 'company_name',
                                            'Issue_Size(crores)' : 'issue_size_crores',
                                            'Offer Price' : 'issue_price',
                                            'List Price' : 'listing_price',
                                            'Listing Gain' : 'listing_gain_pct_source'
                                          })
df_filtered.columns.tolist()

['listing_date',
 'company_name',
 'issue_size_crores',
 'QIB',
 'HNI',
 'RII',
 'Total',
 'issue_price',
 'listing_price',
 'listing_gain_pct_source']

In [6]:
#rows where company_name is truncated coz of source 
df_filtered['name_truncated'] = df_filtered['company_name'].str.endswith('...')
print(df_filtered['name_truncated'].value_counts())

# Check the character length pattern for truncated names
truncated_lengths = df_filtered.loc[df_filtered['name_truncated'], 'company_name'].str.len()
print("\nLength distribution of truncated names:")
print(truncated_lengths.describe())

# Check length of NON-truncated names, for comparison
clean_lengths = df_filtered.loc[~df_filtered['name_truncated'], 'company_name'].str.len()
print("\nLength distribution of clean (non-truncated) names:")
print(clean_lengths.describe())


name_truncated
False    361
True      49
Name: count, dtype: int64

Length distribution of truncated names:
count    49.0
mean     21.0
std       0.0
min      21.0
25%      21.0
50%      21.0
75%      21.0
max      21.0
Name: company_name, dtype: float64

Length distribution of clean (non-truncated) names:
count    361.000000
mean      26.476454
std        8.738810
min        8.000000
25%       20.000000
50%       26.000000
75%       31.000000
max       64.000000
Name: company_name, dtype: float64


In [7]:
# Confirm the date threshold for truncated names
truncated_dates = df_filtered.loc[df_filtered['name_truncated'], 'listing_date']
print("Date range of truncated entries:")
print(truncated_dates.min(), "to", truncated_dates.max())

# Sanity check: any truncated names BEFORE that date?
print("\nAny truncated names before 2025-08-18?")
print((truncated_dates < '2025-08-12').sum())

Date range of truncated entries:
2025-08-12 00:00:00 to 2025-12-30 00:00:00

Any truncated names before 2025-08-18?
0


In [8]:
import requests
import io

nse_url = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"
#creates a headers and session object
headers = {'User-Agent' : 'Mozilla/5.0'}
session = requests.Session()
session.headers.update(headers)
#creates a response object -  send request - get reply as response object
response = session.get(nse_url)
print(response.status_code)

nse_symbols = pd.read_csv(io.BytesIO(response.content))
print(nse_symbols.shape)
nse_symbols.head()

200
(2381, 8)


,SYMBOL,NAME OF COMPANY,SERIES,DATE OF LISTING,PAID UP VALUE,MARKET LOT,ISIN NUMBER,FACE VALUE
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,5,1,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,EQ,03-MAY-1995,10,1,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1,1,INE466L01038,1
3,3BBLACKBIO,3B Blackbio Dx Limited,EQ,20-APR-2026,10,1,INE994E01018,10
4,3IINFOLTD,3i Infotech Limited,EQ,22-OCT-2021,10,1,INE748C01038,10


In [9]:
nse_symbols.to_csv('../data/raw/nse_equity_list.csv', index=False)

In [10]:
!pip install rapidfuzz


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
#removing whitespace from all column names
nse_symbols.columns = nse_symbols.columns.str.strip()
print(nse_symbols.columns.tolist())

['SYMBOL', 'NAME OF COMPANY', 'SERIES', 'DATE OF LISTING', 'PAID UP VALUE', 'MARKET LOT', 'ISIN NUMBER', 'FACE VALUE']


In [12]:
nse_symbols['DATE OF LISTING'] = pd.to_datetime(nse_symbols['DATE OF LISTING'], format = '%d-%b-%Y')
nse_symbols.info()

<class 'pandas.DataFrame'>
RangeIndex: 2381 entries, 0 to 2380
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   SYMBOL           2381 non-null   str           
 1   NAME OF COMPANY  2381 non-null   str           
 2   SERIES           2381 non-null   str           
 3   DATE OF LISTING  2381 non-null   datetime64[us]
 4   PAID UP VALUE    2381 non-null   int64         
 5   MARKET LOT       2381 non-null   int64         
 6   ISIN NUMBER      2381 non-null   str           
 7   FACE VALUE       2381 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 148.9 KB


In [13]:
import re

def normalize_name(name):
    """
    To standardize the comapany names for matching
    -uppercase
    -common words and punctuations
    -extra space
    """
    name = str(name).upper()
    #name = name.replace('-',' ')
    name = re.sub(r'[.,\'"\-()]',' ',name) #remove punctuations
    name = re.sub(r'\b(LIMITED|LTD|LT|L|PRIVATE|PVT)\b','',name) #remove common words
    name = re.sub(r'\s+',' ',name).strip() #remove extra space
    return name

In [14]:
nse_symbols['name_normalized'] = nse_symbols['NAME OF COMPANY'].apply(normalize_name)
df_filtered['name_normalized'] = df_filtered['company_name'].apply(normalize_name)

print(df_filtered[['name_normalized','company_name']].head(10))
print(nse_symbols[['name_normalized','NAME OF COMPANY']].head(10))


      name_normalized           company_name
0  GUJARAT KIDNEY & S  Gujarat Kidney & S...
1   KSH INTERNATIONAL  KSH International ...
2  ICICI PRUDENTIAL A  ICICI Prudential A...
3   NEPHROCARE HEALTH  Nephrocare Health ...
4     PARK MEDI WORLD  Park Medi World Lt...
5  WAKEFIT INNOVATION  Wakefit Innovation...
6     CORONA REMEDIES  Corona Remedies Lt...
7               AEQUS              Aequs Ltd
8              MEESHO             Meesho Ltd
9         VIDYA WIRES        Vidya Wires Ltd
                    name_normalized                           NAME OF COMPANY
0                        20 MICRONS                        20 Microns Limited
1  21ST CENTURY MANAGEMENT SERVICES  21st Century Management Services Limited
2                       360 ONE WAM                       360 ONE WAM LIMITED
3                    3B BLACKBIO DX                    3B Blackbio Dx Limited
4                       3I INFOTECH                       3i Infotech Limited
5                          3M INDIA  

In [15]:
from rapidfuzz import process, fuzz

def find_best_match (name, choices):
    """
    Find the best fuzzy match for the ipo name among the choices from nse symbol list
    and return the name and the score
    """
    result = process.extractOne(name, choices, scorer=fuzz.token_set_ratio) #process used to search through list and fuzz to get the score
    if result:
        return result[0],result[1] #0-best match,1-score,2-index
    return None, 0

In [16]:
#list of choices by NSE 
nse_choices = nse_symbols['name_normalized'].tolist()

#matching every row in the ipo table
matches = df_filtered['name_normalized'].apply(lambda x: find_best_match (x, nse_choices))
df_filtered['matched_name'] = matches.apply(lambda x: x[0])
df_filtered['match_score'] = matches.apply(lambda x: x[1])

df_filtered[['company_name','matched_name','match_score']].head(15)

,company_name,matched_name,match_score
0,Gujarat Kidney & S...,GUJARAT KIDNEY AND SUPER SPECIALITY,87.500000
1,KSH International ...,KSH INTERNATIONAL,100.000000
2,ICICI Prudential A...,ICICI PRUDENTIAL ASSET MANAGEMENT COMPANY,94.117647
3,Nephrocare Health ...,NEPHROCARE HEALTH SERVICES,100.000000
4,Park Medi World Lt...,PARK MEDI WORLD,100.000000
5,Wakefit Innovation...,WAKEFIT INNOVATIONS,97.297297
6,Corona Remedies Lt...,CORONA REMEDIES,100.000000
7,Aequs Ltd,AEQUS,100.000000
8,Meesho Ltd,MEESHO,100.000000
9,Vidya Wires Ltd,VIDYA WIRES,100.000000


In [17]:
#matching and checking score on name basis
print(df_filtered['match_score'].describe())

for threshold in [95,90,85,80,75,70]:
    count = (df_filtered['match_score'] < threshold).sum()
    print(f"Below {threshold} : {count} rows")

count    410.000000
mean      97.822785
std        6.332565
min       55.813953
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: match_score, dtype: float64
Below 95 : 56 rows
Below 90 : 36 rows
Below 85 : 26 rows
Below 80 : 13 rows
Below 75 : 6 rows
Below 70 : 4 rows


In [18]:
def find_best_match_full (name, nse_df):
    choices = nse_df['name_normalized'].to_list()
    result = process.extractOne(name,choices,scorer = fuzz.token_set_ratio)

    if result:
        matched_name,match_score,index = result 
        matched_row = nse_df.iloc[index]
        return matched_row['SYMBOL'],matched_row['name_normalized'],matched_row['DATE OF LISTING'],match_score
    return None,None,None,0        

In [19]:
results = df_filtered['name_normalized'].apply(lambda x:find_best_match_full (x,nse_symbols))
df_filtered['matched_symbol'] = results.apply(lambda x:x[0])
df_filtered['matched_name'] = results.apply(lambda x:x[1])
df_filtered['matched_date_nse'] = results.apply(lambda x:x[2])
df_filtered['match_symbol'] = results.apply(lambda x:x[3])

df_filtered['dod'] = (df_filtered['matched_date_nse'] - df_filtered['listing_date']).dt.days.abs() #difference of dates

df_filtered.head(10)

,listing_date,company_name,issue_size_crores,QIB,HNI,RII,Total,issue_price,listing_price,listing_gain_pct_source,name_truncated,name_normalized,matched_name,match_score,matched_symbol,matched_date_nse,match_symbol,dod
0,2025-12-30,Gujarat Kidney & S...,250.80,1.06,5.73,19.04,5.21,114,120.75,5.92,True,GUJARAT KIDNEY & S,GUJARAT KIDNEY AND SUPER SPECIALITY,87.500000,GKSL,2025-12-30,87.500000,0
1,2025-12-23,KSH International ...,644.45,1.06,0.42,0.86,0.83,384,370.00,-3.65,True,KSH INTERNATIONAL,KSH INTERNATIONAL,100.000000,KSHINTL,2025-12-23,100.000000,0
2,2025-12-19,ICICI Prudential A...,10602.70,123.87,22.04,2.53,39.17,2165,2606.20,20.38,True,ICICI PRUDENTIAL A,ICICI PRUDENTIAL ASSET MANAGEMENT COMPANY,94.117647,ICICIAMC,2025-12-19,94.117647,0
3,2025-12-17,Nephrocare Health ...,871.05,27.47,24.27,2.31,13.96,460,491.70,6.89,True,NEPHROCARE HEALTH,NEPHROCARE HEALTH SERVICES,100.000000,NEPHROPLUS,2025-12-17,100.000000,0
4,2025-12-17,Park Medi World Lt...,920.00,11.48,15.15,3.16,8.10,162,155.60,-3.95,True,PARK MEDI WORLD,PARK MEDI WORLD,100.000000,PARKHOSPS,2025-12-17,100.000000,0
5,2025-12-15,Wakefit Innovation...,1288.89,3.04,1.05,3.17,2.52,195,0.00,NaN,True,WAKEFIT INNOVATION,WAKEFIT INNOVATIONS,97.297297,WAKEFIT,2025-12-15,97.297297,0
6,2025-12-15,Corona Remedies Lt...,655.37,278.52,208.88,28.73,137.04,1062,1452.00,36.72,True,CORONA REMEDIES,CORONA REMEDIES,100.000000,CORONA,2025-12-15,100.000000,0
7,2025-12-10,Aequs Ltd,922.01,120.92,80.62,78.05,101.63,124,140.00,12.90,False,AEQUS,AEQUS,100.000000,AEQUS,2025-12-10,100.000000,0
8,2025-12-10,Meesho Ltd,5421.20,120.18,38.15,19.04,79.02,111,161.20,45.23,False,MEESHO,MEESHO,100.000000,MEESHO,2025-12-10,100.000000,0
9,2025-12-10,Vidya Wires Ltd,300.01,5.12,51.98,27.86,26.59,52,52.13,0.25,False,VIDYA WIRES,VIDYA WIRES,100.000000,VIDYAWIRES,2025-12-10,100.000000,0


In [20]:
print(df_filtered['match_score'].describe())
print("\nDate diff distribution:")
print(df_filtered['dod'].describe())

count    410.000000
mean      97.822785
std        6.332565
min       55.813953
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: match_score, dtype: float64

Date diff distribution:
count      410.000000
mean       342.670732
std       1435.726968
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      10380.000000
Name: dod, dtype: float64


In [21]:
# These are REITs/InvITs (not equity IPOs) or companies since delisted - excluding from study
drop_companies = [
    'Embassy Office Parks REIT',       # REIT
    'Nexus Select Trust',               # REIT
    'Mindspace Business Parks REIT',    # REIT
    'Indus Infra Trust',                # InvIT
    'Brookfield India Real Estate Trust', # REIT
    'Powergrid Infrastructure Investment Trust', # InvIT
    'ICICI Securities Ltd',             # delisted 2024 (ICICI Bank buyback)
    'TCNS Clothing Co. Ltd',             # merged into Aditya Birla Fashion
    'Sah Polymers Limited',              # renamed/merged
    'National Securities Depository Ltd (NSDL)',  # no confirmed NSE match
]
df_filtered = df_filtered[~df_filtered['company_name'].str.strip().isin(drop_companies)].copy()
df_filtered.shape

(400, 18)

In [22]:
accept = ((df_filtered['match_score'] >= 90) | (df_filtered['dod'] < 3))
print(f"Auto Accepted matches are {accept.sum()}/{len(df_filtered)}")
print(f"Manual review is required {(len(df_filtered) - accept.sum())}/{len(df_filtered)} matches")

Auto Accepted matches are 391/400
Manual review is required 9/400 matches


In [23]:
#Manual Review
needs_review = df_filtered[~accept].sort_values('match_score')

print(f"The no. of rows to be reviewed is :{len(needs_review)}")
manual_review = needs_review[['company_name','matched_name','matched_symbol','match_score','listing_date','matched_date_nse','dod']]
manual_review

The no. of rows to be reviewed is :9


,company_name,matched_name,matched_symbol,match_score,listing_date,matched_date_nse,dod
92,Schloss Bangalore Limited,ORISSA BENGAL CARRIER,OBCL,57.894737,2025-06-02,2022-04-07,1152
335,Barbeque Nation Hospitality Limited,GRAVISS HOSPITALITY,GRAVISSHO,73.333333,2021-04-07,2026-04-20,1839
217,Signature Global India Limited,3M INDIA,3MINDIA,76.923077,2023-09-27,2004-08-13,6984
376,Affle (India) Ltd,SCHAEFFLER INDIA,SCHAEFFLER,81.481481,2019-08-08,2000-11-29,6826
99,Dr Agarwal's Healthcare Limited,GPT HEALTHCARE,GPTHEALTH,83.333333,2025-02-04,2024-02-29,341
63,Bluestone Jeweller...,PC JEWELLER,PCJEWELLER,84.210526,2025-08-19,2012-12-27,4618
96,Quality Power Electronics Limited,GNG ELECTRONICS,EBGNG,84.615385,2025-02-24,2025-07-30,156
287,AGS Transact Technologies Limited,AAA TECHNOLOGIES,AAATECH,85.714286,2022-01-31,2022-11-28,301
100,Denta Water & Infrastructure Limited,GTL INFRASTRUCTURE,GTLINFRA,87.500000,2025-01-29,2006-11-09,6656


In [24]:
manual_review.to_csv('../data/processed/needs_manual_review.csv', index=False)

In [25]:
manual_symbol_overrides = {
    'Signature Global India Limited': 'SIGNATURE',
    'Quality Power Electronics Limited': 'QPOWER',
    'Denta Water & Infrastructure Limited': 'DENTA',
    'Barbeque Nation Hospitality Limited': 'UFBL',
    "Dr Agarwal's Healthcare Limited": 'AGARWALEYE',
    'AGS Transact Technologies Limited': 'AGSTRA',
    'Schloss Bangalore Limited': 'THELEELA',
    'Affle (India) Ltd' : 'AFFLE',   
    'Bluestone Jeweller...' : 'BLUESTONE' 
}

for company, symbol in manual_symbol_overrides.items() :
    df_filtered.loc[df_filtered['company_name'].str.strip() == company,'matched_symbol'] = symbol

df_filtered[df_filtered['company_name'].isin(manual_symbol_overrides.keys())][['company_name','matched_symbol']] 

,company_name,matched_symbol
63,Bluestone Jeweller...,BLUESTONE
92,Schloss Bangalore Limited,THELEELA
96,Quality Power Electronics Limited,QPOWER
99,Dr Agarwal's Healthcare Limited,AGARWALEYE
100,Denta Water & Infrastructure Limited,DENTA
217,Signature Global India Limited,SIGNATURE
287,AGS Transact Technologies Limited,AGSTRA
335,Barbeque Nation Hospitality Limited,UFBL
376,Affle (India) Ltd,AFFLE


In [26]:
print(df_filtered.shape)
print(df_filtered[['company_name', 'matched_symbol', 'issue_price', 'listing_price', 'listing_date']].isnull().sum())

# Also confirm no duplicate symbols (would indicate a matching error)
print("\nDuplicate matched_symbol check:")
print(df_filtered['matched_symbol'].duplicated().sum())

(400, 18)
company_name      0
matched_symbol    0
issue_price       0
listing_price     0
listing_date      0
dtype: int64

Duplicate matched_symbol check:
2


In [27]:
dup_symbols = df_filtered[df_filtered['matched_symbol'].duplicated(keep=False)].sort_values('matched_symbol')
dup_symbols[['company_name', 'matched_symbol', 'listing_date', 'match_score']]

,company_name,matched_symbol,listing_date,match_score
284,Patanjali Foods Limited (formerly Ruchi Soya I...,LTFOODS,2022-04-08,100.0
302,Sapphire Foods India,LTFOODS,2021-11-18,100.0
27,Anantam Highways T...,TTL,2025-10-16,100.0
67,Knowledge Realty T...,TTL,2025-08-18,100.0


In [31]:
drop_companies += [
    'Anantam Highways T...',#trusts
    'Knowledge Realty T...',#trusts
    'Patanjali Foods Limited (formerly Ruchi Soya Industries Limited)'  #merged
]

df_filtered = df_filtered[~df_filtered['company_name'].str.strip().isin(drop_companies)].copy()
print(df_filtered.shape)

manual_symbol_overrides['Sapphire Foods India'] = 'SAPPHIRE'  

for company, symbol in manual_symbol_overrides.items():
    df_filtered.loc[df_filtered['company_name'].str.strip() == company, 'matched_symbol'] = symbol
df_filtered[df_filtered['company_name'].str.strip() == 'Sapphire Foods India'][['company_name','matched_symbol']]

(397, 18)


,company_name,matched_symbol
302,Sapphire Foods India,SAPPHIRE


In [32]:
print(df_filtered.shape)
print(df_filtered[['company_name', 'matched_symbol', 'issue_price', 'listing_price', 'listing_date']].isnull().sum())

# Also confirm no duplicate symbols (would indicate a matching error)
print("\nDuplicate matched_symbol check:")
print(df_filtered['matched_symbol'].duplicated().sum())

(397, 18)
company_name      0
matched_symbol    0
issue_price       0
listing_price     0
listing_date      0
dtype: int64

Duplicate matched_symbol check:
0


In [35]:
final_columns = ['company_name', 'matched_symbol', 'listing_date', 'issue_price','listing_price', 'issue_size_crores', 'QIB', 'HNI', 'RII', 'Total']

df_final = df_filtered[final_columns].copy()
df_final = df_final.rename(columns = {'matched_symbol':'nse_symbol'})

df_final.to_csv('../data/processed/ipo_clean.csv', index=False)

print(df_final.shape)
df_final.head()


(397, 10)


,company_name,nse_symbol,listing_date,issue_price,listing_price,issue_size_crores,QIB,HNI,RII,Total
0,Gujarat Kidney & S...,GKSL,2025-12-30,114,120.75,250.80,1.06,5.73,19.04,5.21
1,KSH International ...,KSHINTL,2025-12-23,384,370.00,644.45,1.06,0.42,0.86,0.83
2,ICICI Prudential A...,ICICIAMC,2025-12-19,2165,2606.20,10602.70,123.87,22.04,2.53,39.17
3,Nephrocare Health ...,NEPHROPLUS,2025-12-17,460,491.70,871.05,27.47,24.27,2.31,13.96
4,Park Medi World Lt...,PARKHOSPS,2025-12-17,162,155.60,920.00,11.48,15.15,3.16,8.10


## Step 1 Summary: Data Collection & Ticker Matching

Started with 410 mainboard IPOs (2018–2025) from a Kaggle IPO dataset. Matched 
company names to NSE ticker symbols using fuzzy matching (rapidfuzz, 
token_set_ratio), cross-validated against listing dates from NSE's official 
equity list.

**Key issues found and resolved:**
- Company names truncated to 21 characters for listings after 2025-08-12 
  in the source data — handled via fuzzy matching on partial names
- Filtering the NSE pool by SERIES == 'EQ' silently excluded valid companies 
  temporarily under surveillance-related series (BE, etc.) — removed this filter
- Hyphen-stripping in name normalization merged compound words incorrectly 
  (e.g. "Infra-EPC" → "InfraEPC") — fixed by replacing hyphens with spaces
- 13 rows excluded as non-IPO events: REITs/InvITs (8), a delisted company 
  (ICICI Securities), a company merger (TCNS Clothing), an FPO-only listing 
  (Vodafone Idea), and a post-insolvency relisting (Patanjali Foods/Ruchi Soya)
- 10 rows required manual symbol overrides due to company renames 
  (e.g. Barbeque Nation Hospitality → United Foodbrands) or wording 
  mismatches between source datasets

**Final clean dataset:** 397 mainboard IPOs with verified NSE ticker symbols.